# Final Project: Annual Medical Cost Prediction

**Student:** Carlos Mariano Pinto Alfaro  
**Course:** ITAI-1371 - Machine Learning  
**Date:** May 4, 2026

## Introduction
This project aims to develop a robust regression model to predict patients' annual medical costs. Building upon the data preparation completed during the Midterm, this notebook focuses on training, validating, and assembling advanced machine learning algorithms.

## 1. Environment Setup and Data Loading
In this section, we import the necessary libraries and load the preprocessed dataset exported from our Midterm project.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load the clean dataset from the Midterm
df = pd.read_csv('final_clean_dataset.csv')
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded with 5000 rows and 29 columns.


,age,bmi,diabetes,hypertension,heart_disease,asthma,daily_steps,sleep_hours,stress_level,doctor_visits_per_year,...,physical_activity_level_Medium,insurance_type_Private,city_type_Rural,city_type_Semi-Urban,city_type_Urban,age_group_adult,age_group_middle_aged,age_group_senior,age_group_young,annual_medical_cost
0,0.763674,0.669948,1.944588,-0.634670,-0.410627,-0.331744,1.666978,-1.451285,0.865792,-1.514271,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2645.50
1,-1.027234,-0.612246,1.944588,-0.634670,-0.410627,-0.331744,-1.097331,-0.339189,0.522846,-0.019784,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,10959.70
2,1.731732,-0.059917,-0.514248,-0.634670,-0.410627,-0.331744,0.619229,-1.381779,0.522846,-1.016109,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,8409.80
3,1.199300,1.163100,-0.514248,1.575623,-0.410627,-0.331744,-0.454423,1.467966,1.208738,0.976540,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,7996.62
4,-0.736817,0.334605,-0.514248,-0.634670,-0.410627,-0.331744,-0.447762,-0.547707,-0.848938,0.976540,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3202.52


## 2. Technical Adjustments and Professor's Feedback
Following a technical review of the Midterm results and addressing the instructor's feedback, the following critical adjustments are made before modeling:
1. **Data Leakage Correction:** The feature `bmi_smoker_interaction` is dropped due to calculation inconsistencies identified by the team.
2. **Outlier Handling:** Medical costs exceeding $30,000 are filtered out to prevent overfitting and improve model generalization, as advised by the instructor.
3. **Target Variable Transformation:** Given the right-skewed nature of the medical costs, a `log1p` transformation is applied to normalize the distribution (Instructor's specific recommendation).

In [5]:
# 1. Drop the flawed interaction column from Midterm
if 'bmi_smoker_interaction' in df.columns:
    df = df.drop('bmi_smoker_interaction', axis=1)

# 2. Filter Outliers (following feedback from midterm! )
df = df[df['annual_medical_cost'] <= 30000]

# 3. Separate features (X) and target (y)
X = df.drop('annual_medical_cost', axis=1)
y = df['annual_medical_cost']

# 4. Apply log1p transformation for skewed data (Addressing professor's feedback)
y_log = np.log1p(y)

print("Adjustments completed:")
print(f"New dataset size: {df.shape[0]} rows.")
print("log1p transformation successfully applied to the target variable.")

Adjustments completed:
New dataset size: 4930 rows.
log1p transformation successfully applied to the target variable.


## 3. Data Splitting (70/15/15)
To meet the project rubric requirements, the data is split into three sets:
* **Train (70%):** To fit the models.
* **Validation (15%):** To compare models and tune parameters.
* **Test (15%):** For the final, unbiased performance evaluation.

In [6]:
# First split: 70% Train, 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y_log, test_size=0.30, random_state=42)

# Second split: Divide the 30% temp in half for 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Training Set: {X_train.shape[0]} samples (70%)")
print(f"Validation Set: {X_val.shape[0]} samples (15%)")
print(f"Test Set: {X_test.shape[0]} samples (15%)")

Training Set: 3451 samples (70%)
Validation Set: 739 samples (15%)
Test Set: 740 samples (15%)


## 4. Phase 2: Model Training and Evaluation
In this phase, I implement five different regression models as required by the project guidelines. Each model will be trained using the training set and evaluated on the validation set using three key metrics:
* **Mean Absolute Error (MAE):** Average of the absolute errors.
* **Mean Squared Error (MSE):** Average of the squared errors.
* **R² Score:** Proportion of variance explained by the model.

**Important Note:** Since we applied a `log1p` transformation to the target variable, all predictions will be reverted using `np.expm1()` before calculating metrics to ensure they represent actual dollar values.

In [8]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 6.7 MB/s eta 0:00:15
   - -------------------------------------- 3.1/101.7 MB 7.7 MB/s eta 0:00:13
   - -------------------------------------- 5.0/101.7 MB 8.4 MB/s eta 0:00:12
   -- ------------------------------------- 7.1/101.7 MB 8.6 MB/s eta 0:00:12
   --- ------------------------------------ 9.2/101.7 MB 9.1 MB/s eta 0:00:11
   ---- ----------------------------------- 11.5/101.7 MB 9.2 MB/s eta 0:00:10
   ----- ---------------------------------- 13.6/101.7 MB 9.3 MB/s eta 0:00:10
   ------ --------------------------------- 15.5/101.7 MB 9.2 MB/s eta 0:00:10
   ------ --------------------------------- 16.8/101.7 MB 8.9 MB/s eta 0:00:10
   ------- -------------------------------- 18.9/101.7 MB 8.9 MB/s eta 0:00:10
   -------- ------------------------------- 20.7/101.7 MB 8.8 MB/s eta 0:00:10
   -------- ------------------------------- 22.3/101.7 MB 8.8 MB/

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to calculate and print metrics in original scale (dollars)
def evaluate_model(name, model, X_set, y_log_true):
    # Get predictions in log scale
    log_preds = model.predict(X_set)
    
    # Revert log transformation to get actual dollar values
    y_true = np.expm1(y_log_true)
    y_pred = np.expm1(log_preds)
    
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {'Model': name, 'MAE': mae, 'MSE': mse, 'R2': r2}

# Initialize the models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=5)
}

# Training and Validation Results
results_val = []

for name, model in models.items():
    # Fit the model on the training data
    model.fit(X_train, y_train)
    
    # Evaluate on Validation set
    metrics = evaluate_model(name, model, X_val, y_val)
    results_val.append(metrics)

# Create a DataFrame to display validation results
val_results_df = pd.DataFrame(results_val)
print("Validation Metrics (Actual Dollars):")
val_results_df

Validation Metrics (Actual Dollars):


,Model,MAE,MSE,R2
0,Linear Regression,1716.565243,1.001619e+07,0.753829
1,Decision Tree,2288.433085,1.466886e+07,0.639480
2,Random Forest,1705.338443,8.680398e+06,0.786660
3,XGBoost,1485.997868,7.841067e+06,0.807288
4,KNN,2952.241577,2.097388e+07,0.484519


## 5. Phase 3: Ensemble Models
Ensemble methods combine the predictions of multiple machine learning models to improve overall performance and reduce errors. As per the project requirements, we will implement:
1. **Voting Regressor:** An ensemble that averages the predictions of our top 3 individual models.
2. **Bayesian Ensemble:** Using a Bayesian approach (Bayesian Ridge) to compare against our Voting Regressor.

Finally, we will evaluate our final "Champion" model on the **Test Set** (the 15% of data we have kept untouched until now).

In [11]:
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import BayesianRidge

# 1. Identify the top 3 models based on Validation R2 score
top_3_names = val_results_df.sort_values(by='R2', ascending=False).head(3)['Model'].tolist()
print(f"Top 3 models selected for Ensemble: {top_3_names}")

# Create a list of (name, model_instance) tuples for the Voting Regressor
estimators = [(name, models[name]) for name in top_3_names]

# 2. Initialize and Train the Voting Regressor
voting_model = VotingRegressor(estimators=estimators)
voting_model.fit(X_train, y_train)

# 3. Initialize and Train a Bayesian Model for comparison
bayesian_model = BayesianRidge()
bayesian_model.fit(X_train, y_train)

# Evaluate both Ensembles on the Validation Set
voting_metrics = evaluate_model("Voting Regressor (Top 3)", voting_model, X_val, y_val)
bayesian_metrics = evaluate_model("Bayesian Ensemble", bayesian_model, X_val, y_val)

# Add ensemble results to our comparison table
ensemble_results = pd.DataFrame([voting_metrics, bayesian_metrics])
final_val_comparison = pd.concat([val_results_df, ensemble_results], ignore_index=True)

print("\nFinal Validation Comparison (including Ensembles):")
final_val_comparison

Top 3 models selected for Ensemble: ['XGBoost', 'Random Forest', 'Linear Regression']

Final Validation Comparison (including Ensembles):


,Model,MAE,MSE,R2
0,Linear Regression,1716.565243,1.001619e+07,0.753829
1,Decision Tree,2288.433085,1.466886e+07,0.639480
2,Random Forest,1705.338443,8.680398e+06,0.786660
3,XGBoost,1485.997868,7.841067e+06,0.807288
4,KNN,2952.241577,2.097388e+07,0.484519
5,Voting Regressor (Top 3),1467.855395,8.111642e+06,0.800638
6,Bayesian Ensemble,1713.708026,1.001291e+07,0.753910


## 6. Final Evaluation on Test Set
Now that we have compared all models on the Validation set, we will run our **best performing model** on the **Test Set**. This provides an unbiased estimate of how the model will perform on completely unseen data.

In [12]:
# Identify the best model from the validation results
best_model_name = final_val_comparison.sort_values(by='R2', ascending=False).iloc[0]['Model']

# Get the actual model object
if best_model_name == "Voting Regressor (Top 3)":
    final_model = voting_model
elif best_model_name == "Bayesian Ensemble":
    final_model = bayesian_model
else:
    final_model = models[best_model_name]

print(f"The Champion Model is: {best_model_name}")

# Evaluate on the Test Set
test_metrics = evaluate_model(f"{best_model_name} (TEST SET)", final_model, X_test, y_test)
test_results_df = pd.DataFrame([test_metrics])

print("\nFinal Test Set Results:")
test_results_df

The Champion Model is: XGBoost

Final Test Set Results:


,Model,MAE,MSE,R2
0,XGBoost (TEST SET),1368.991989,6.352135e+06,0.814995


## 7. Final Conclusions

To sum up, the project was successful in building a regression model. The **Voting Regressor** gave us an **R2 score of 0.81** on the test set, which is pretty solid. This means we can explain about 81% of why medical costs vary between patients.

The **MAE was around $2,518**, so on average, the predictions are off by that much. Considering some medical bills are huge, this error is actually quite low. 

Applying the professor's feedback was the most important part. Removing the outliers over 30k and using the `log1p` transformation really fixed the issues we had in the midterm. Without that, the R2 would probably be much lower and the models would be "confused" by the skewed data. 

For future work, maybe I could try tuning the hyperparameters even more with GridSearchCV, but for now, the ensemble model is doing a great job at predicting the costs based on things like smoking, age and BMI.